# Phosphorylation dynamics clustering and QC

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.compat.pandas import to_numpy

from utils import *

In [ ]:
df_2_hME1_scaled = pd.read_csv("Experiment/2_hTERT_HME1/Data/Processed/Full_dataset_2_hTERT_HME1_functional_names_diff_scaled_phPlus.tsv", sep="\t")
df_2_hME1_scaled = df_2_hME1_scaled.fillna(0)

In [ ]:
print(f"The Dataframe shape: {df_2_hME1_scaled.shape}")
df_2_hME1_scaled.head()

## Filter dataframe
- More than 1 replicate
- Localized phosphorylation site = True
- log2FC higher than 0.5 (excluding the full time point)

In [ ]:
nreps_df = filter_replicates(df_2_hME1_scaled, n_reps=2)
localized_sites_df = filter_site_localizations(nreps_df, loc_sites=True)
df_filtered = filter_dynamics_extremes(df = localized_sites_df,
                                       data_type="log2_FC",
                                       threshold=0.5,
                                       exclude_full=True)
print(f"Original Dataframe shape: {df_2_hME1_scaled.shape}"
      f"\nnreps_df: {nreps_df.shape}"
      f"\nlocalized_sites_df: {localized_sites_df.shape}"
      f"\ndf_filtered: {df_filtered.shape}")

## Cluster filtered dataframe using tslearn KMeans. Exclude "full" for the clustering

**Using tslearn.KMeans there is NO DIFFERENCE between clustering the dataset as (n, 3, 7) or (n, 7, 3)**

In [ ]:
cluster_name = "KMeans_25_cluster_on_log2FC_noFull_nrep>1_locSite=True_log2FC>0.5"
kmeans_clustered = tslearn_clustering_KMeans(df_to_cluster= df_filtered,
                                             data_type= "log2_FC",
                                             condition_for_clustering=["_EGF_","_INS_","_EGFnINS_",],
                                             exclude_full=True,
                                             cluster_column_name=cluster_name,
                                             number_of_clusters=25,
                                             max_iterations=100,
                                             metric="euclidean",
                                             df_dimensions=3,
                                             time_series_length=6,
                                             transpose = True, #data size is (ns, sz, d)
                                             verbose=True)

In [ ]:
clusters_plot(df = kmeans_clustered,
              legend=["1.57nM EGF", "100nM INS", "1.56nM EGF + 100nM INS"],
              saving_path = "",
              cluster_column = cluster_name,
              cluster_name = "Additional information",
              data_type="log2_FC",
              plot_different_data=True,
              saving_info = "",
              save_pdf=False,
              save_png = False,
              plot_close = False,
              y_lims_list=False)

In [ ]:
cluster_number = 15
plot_dataset_phosphosites(df = kmeans_clustered,
                          cluster_column = cluster_name,
                          cluster_number = cluster_number,
                          data_type = "log2_FC",
                          legend=['1.57 nM EGF', '100 nM Ins', '1.57 nM EGF + 100 nM Ins'],
                          color_palette = ['r', 'b', 'fuchsia'],
                          saving_path="",
                          dataset_name="",
                          saving_info="",
                          plot_individually=False,
                          fit_y_lims=False,
                          save_pdf=False,
                          save_png=False,
                          plot_close=False
                          )
uniprot_links_for(kmeans_clustered, protein_list=list(kmeans_clustered.loc[kmeans_clustered[cluster_name] == cluster_number]["protein_name"].unique()))

In [ ]:
cluster_QC = cluster_similarity_cdist_dtw(df=kmeans_clustered,
                                          transpose=False,
                                          data_type="log2_FC",
                                          cluster_column_name=cluster_name,
                                          mean=True,
                                          median=False,
                                          verbose = True)
plt.bar(x = cluster_QC.keys(), height = cluster_QC.values())

In [ ]:
np.mean(list(cluster_QC.values()))

In [ ]:
cluster_QC_test = cluster_similarity_per_condition(df=kmeans_clustered,
                                                   transpose=True,
                                                   data_type="FC_scaled",
                                                   cluster_column_name="KMeans_25_cluster_on_log2FC_noFull_nrep>1_locSite=True_log2FC>0.5",
                                                   mean=True,
                                                   median=False,
                                                   verbose = False)

plot_grouped_bars(cluster_QC_test)

In [ ]:
values = [v for inner in cluster_QC_test.values() for v in inner.values()]
np.mean(values)

In [ ]:
cluster_QC_test_per_timepoint = cluster_similarity_per_condition_per_timepoint(df=kmeans_clustered,
                                                   transpose=True,
                                                   data_type="FC_scaled",
                                                   cluster_column_name="KMeans_25_cluster_on_log2FC_noFull_nrep>1_locSite=True_log2FC>0.5",
                                                   mean=True,
                                                   median=False,
                                                   verbose = False)
timepoints = [-3, 0, 2, 5, 10, 15, 90]
plot_cluster_timepoint_dispersion(cluster_QC_test_per_timepoint, ncols=5)

In [ ]:
overall_score = combine_conditions(cluster_QC_test_per_timepoint, how="mean")

plt.bar(x=overall_score.keys(), height=overall_score.values())

np.mean(list(overall_score.values()))

### Clustering KMeans, testing fit_transform()

In [ ]:
# nk = 7
for nk in range(7, 8):
    cluster_name = f"KMeans_{nk}_cluster_on_log2FC_noFull_nrep>1_locSite=True_log2FC>0.5_transform_only_EGF"
    kmeans_clustered, clustering, multivariate, barycenters = tslearn_clustering_KMeans(df_to_cluster= df_filtered,
                                                 data_type= "log2_FC",
                                                 condition_for_clustering=["_EGF_"],
                                                 exclude_full=True,
                                                 cluster_column_name=cluster_name,
                                                 number_of_clusters=nk,
                                                 max_iterations=100,
                                                 metric="euclidean",
                                                 df_dimensions=1,
                                                 time_series_length=6,
                                                 transpose = True, #data size is (ns, sz, d)
                                                 verbose=True,
                                                 testing = True,
                                                 barycenter_calculations= True)
    #clustering


    # Darker/low values = closer to that centroid.
    # You want each group of rows (same label) to have a clear minimum in the corresponding column.
    # If rows show similarly low values across many columns → clusters overlap / not well separated.
    order = np.argsort(clustering.labels_)
    sorted = barycenters[order]
    labels_sorted = clustering.labels_[order]

    plt.figure()
    plt.imshow(sorted, aspect="auto")
    plt.xlabel("Cluster (centroid index)")
    plt.ylabel("Samples (sorted by assigned cluster)")
    plt.title("Distances to centroids (sorted by assigned labels)")
    plt.colorbar(label="distance")
    plt.savefig(f"Experiment/2_hTERT_HME1/Results/Clusters_QC/KMeans/EGF_{nk}_clusters_log2FC_distance_to_centroids_heatmap.png")
    plt.show()


    # Visualize assigment confidence. MARGIN  is the distance to the centroid of the next closest cluster to the site, minus the distance to the centroid to which the site was assigned to. Low margin: weak separation, high margin, high separation (the site stringly belong to the cluster). margin between 0.5 and 1 shows, there is some confidence i  the assigment. below 0.3 means weak separation.
    # large margin --> the best centroid is clarely better than the alternatives

    # compute per-sample margin (2nd best - best)
    sorted_D = np.sort(barycenters, axis=1)
    margin = sorted_D[:, 1] - sorted_D[:, 0]

    plt.figure()
    plt.hist(margin, bins=30)
    plt.xlabel("Margin = 2nd_best - best")
    plt.ylabel("Count")
    plt.title("Cluster assignment margin distribution")
    plt.savefig(f"Experiment/2_hTERT_HME1/Results/Clusters_QC/KMeans/EGF_{nk}_clusters_log2FC_Cluster_assignment_margin_distribution.png")
    plt.show()

    plt.figure()
    plt.boxplot([margin[clustering.labels_ == k] for k in range(barycenters.shape[1])]) #, tick_labels=[str(k) for k in range(clustering.shape[1])])
    plt.xlabel("Cluster")
    plt.ylabel("Margin")
    plt.title("Assignment margin per cluster")
    plt.savefig(f"Experiment/2_hTERT_HME1/Results/Clusters_QC/KMeans/_EGF{nk}_clusters_log2FC_Assignment_margin_per_cluster.png")
    plt.show()

    # 2D embedding of samples using distances (PCA on the model)
    #I HAVE TO TRY HERE WITH UMAP/T-SNE

    from sklearn.decomposition import PCA

    Z = PCA(n_components=2).fit_transform(barycenters)

    plt.figure()
    plt.scatter(Z[:, 0], Z[:, 1], c=clustering.labels_)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("PCA of distance-to-centroid representation")
    plt.savefig(f"Experiment/2_hTERT_HME1/Results/Clusters_QC/KMeans/{nk}_clusters_log2FC_PCA_distance_to_centroids.png")
    plt.show()

    clusters_plot_linear(
        df = kmeans_clustered,
        legend=["1.57nM EGF", "100nM INS", "1.56nM EGF + 100nM INS"],
        saving_path="Experiment/2_hTERT_HME1/Results/Clusters_QC/KMeans",
        cluster_column=cluster_name,
        cluster_name=f"EGF_{nk}_clusters",
        data_type="log2_FC",
        plot_different_data=True,
        saving_info="_log2FC_Shapes",
        save_pdf=False,
        save_png=True,
        plot_close=False,
        y_lims_list=False,
        grey_alpha=0.2,
        grey_lw=0.8,
        mean_lw=2.6
    )

In [ ]:
# Great for understanding “why did it get assigned to that cluster?”
i = 10# pick a sample index
plt.figure()
plt.bar(np.arange(barycenters.shape[1]), barycenters[i])
plt.xlabel("Cluster")
plt.ylabel("Distance to centroid")
plt.title(f"Sample {i}: distances to centroids (assigned={clustering.labels_[i]})")
plt.show()

In [ ]:
# Look for cluster compactness and separation.
# summary
for k in range(barycenters.shape[1]):
    within = barycenters[clustering.labels_ == k, k]          # distance to assigned centroid
    print(k, "n=", within.size, "mean=", np.mean(within), "median=", np.median(within))

## Cluster filtered dataframe using tslearn Kernel clustering. Exclude "full" for the clustering

time for non-transposed for 8000 sites, 25 clusters -> around 15 mins (15 secs for 1000 sites)

time for transposed for 8000 sites, 25 clusters -> around 17 mins

In [ ]:
kernel_clustered = kernnel_clustering(df = df_filtered,
                                       transpose = False,
                                       data_type = "log2_FC",
                                       exclude_full = True,
                                       condition_for_clustering = ["_EGF_", "_INS_", "EGFnINS"],
                                       df_dimensions = 3,
                                       time_series_length = 6,
                                       seed = 0,
                                       n_clusters = 25,
                                       n_init = 20,
                                       verbose = True,
                                       kernel = "gak",
                                       kernel_params={"sigma": "auto"},
                                       cluster_column_name = "Kernel_cluster_25")

In [ ]:
kernel_clustered_T = kernnel_clustering(df = df_filtered,
                                       transpose = True,
                                       data_type = "log2_FC",
                                       exclude_full = True,
                                       condition_for_clustering = ["_EGF_", "_INS_", "EGFnINS"],
                                       df_dimensions = 3,
                                       time_series_length = 6,
                                       seed = 0,
                                       n_clusters = 25,
                                       n_init = 20,
                                       verbose = True,
                                       kernel = "gak",
                                       kernel_params={"sigma": "auto"},
                                       cluster_column_name = "Kernel_cluster_25_T")

In [ ]:
cluster_kernel_name = "Kernel_cluster_25"
clusters_plot(df = kernel_clustered,
              legend=["1.57nM EGF", "100nM INS", "1.56nM EGF + 100nM INS"],
              saving_path = "",
              cluster_column = cluster_kernel_name,
              cluster_name = "Additional information",
              data_type="log2_FC",
              plot_different_data=True,
              saving_info = "",
              save_pdf=False,
              save_png = False,
              plot_close = False,
              y_lims_list=False)

cluster_kernel_name_T = "Kernel_cluster_25_T"
clusters_plot(df = kernel_clustered_T,
              legend=["1.57nM EGF", "100nM INS", "1.56nM EGF + 100nM INS"],
              saving_path = "",
              cluster_column = cluster_kernel_name_T,
              cluster_name = "Additional information",
              data_type="log2_FC",
              plot_different_data=True,
              saving_info = "",
              save_pdf=False,
              save_png = False,
              plot_close = False,
              y_lims_list=False)

In [ ]:
kernel_cluster_QC = cluster_similarity_cdist_dtw(df=kernel_clustered,
                                          transpose=True,
                                          data_type="FC_scaled",
                                          cluster_column_name=cluster_kernel_name,
                                          mean=True,
                                          median=False,
                                          verbose = False)
plt.bar(x = kernel_cluster_QC.keys(), height = kernel_cluster_QC.values())

In [ ]:
kernel_cluster_QC_T = cluster_similarity_cdist_dtw(df=kernel_clustered_T,
                                          transpose=True,
                                          data_type="FC_scaled",
                                          cluster_column_name=cluster_kernel_name_T,
                                          mean=True,
                                          median=False,
                                          verbose = False)
plt.bar(x = kernel_cluster_QC_T.keys(), height = kernel_cluster_QC_T.values())

In [ ]:
print(np.mean(list(kernel_cluster_QC.values())))
print(np.mean(list(kernel_cluster_QC_T.values())))

In [ ]:
kernel_cluster_QC_per_condition = cluster_similarity_per_condition(df=kernel_clustered,
                                                   transpose=True,
                                                   data_type="log2_FC",
                                                   cluster_column_name=cluster_kernel_name,
                                                   mean=True,
                                                   median=False,
                                                   verbose = False)

plot_grouped_bars(kernel_cluster_QC_per_condition)

In [ ]:
kernel_cluster_QC_per_condition_T = cluster_similarity_per_condition(df=kernel_clustered_T,
                                                   transpose=False,
                                                   data_type="FC_scaled",
                                                   cluster_column_name=cluster_kernel_name_T,
                                                   mean=True,
                                                   median=False,
                                                   verbose = False)

plot_grouped_bars(kernel_cluster_QC_per_condition_T)

In [ ]:
values = [v for inner in kernel_cluster_QC_per_condition.values() for v in inner.values()]
print(np.mean(values))
values_T = [v for inner in kernel_cluster_QC_per_condition_T.values() for v in inner.values()]
print(np.mean(values_T))

In [ ]:
kernel_cluster_QC_per_condition_per_timepoint = cluster_similarity_per_condition_per_timepoint(df=kernel_clustered,
                                                   transpose=True,
                                                   data_type="log2_FC",
                                                   cluster_column_name=cluster_kernel_name,
                                                   mean=True,
                                                   median=False,
                                                   verbose = False)
timepoints = [-3, 0, 2, 5, 10, 15, 90]
plot_cluster_timepoint_dispersion(kernel_cluster_QC_per_condition_per_timepoint, ncols=5)

In [ ]:
kernel_cluster_QC_per_condition_per_timepoint_T = cluster_similarity_per_condition_per_timepoint(df=kernel_clustered_T,
                                                   transpose=True,
                                                   data_type="FC_scaled",
                                                   cluster_column_name=cluster_kernel_name_T,
                                                   mean=True,
                                                   median=False,
                                                   verbose = False)
timepoints = [-3, 0, 2, 5, 10, 15, 90]
plot_cluster_timepoint_dispersion(kernel_cluster_QC_per_condition_per_timepoint_T, ncols=5)

In [ ]:
kernel_cluster_QC_per_condition_per_timepoint_overall = combine_conditions(kernel_cluster_QC_per_condition_per_timepoint, how="mean")

plt.bar(x=kernel_cluster_QC_per_condition_per_timepoint_overall.keys(), height=kernel_cluster_QC_per_condition_per_timepoint_overall.values())

In [ ]:
kernel_cluster_QC_per_condition_per_timepoint_overall_T = combine_conditions(kernel_cluster_QC_per_condition_per_timepoint_T, how="mean")

plt.bar(x=kernel_cluster_QC_per_condition_per_timepoint_overall_T.keys(), height=kernel_cluster_QC_per_condition_per_timepoint_overall_T.values())

In [ ]:
print(np.mean(list(kernel_cluster_QC_per_condition_per_timepoint_overall.values())))
print(np.mean(list(kernel_cluster_QC_per_condition_per_timepoint_overall_T.values())))

## Cluster filtered dataframe using tslearn Kshape. Exclude "full" for the clustering

Only works with dataset shape (n, 21, 1) or (n, 6, 1)


In [ ]:
KShape_all_univariate= tslearn_clustering_KShape(df_to_cluster = df_filtered,
                              data_type = "log2_FC",
                              condition_for_clustering = ["_EGF_", "_INS_", "EGFnINS"],
                              exclude_full = False,
                              number_of_clusters = 25,
                              cluster_column_name = "KShape_cluster_25_log2_FC_all",
                              max_iterations = 1000,
                              # metric='euclidean',
                              df_dimensions = 1,
                              time_series_length = 21,
                              transpose = True,
                              verbose = True )
KShape_all_univariate[["site","KShape_25_log2_FC_all"]]

In [ ]:
KShape_all_univariate.KShape_25_log2_FC_all.unique()

In [ ]:
cluster_name= "KShape_cluster_25_log2_FC_all"
clusters_plot(df = KShape_all_univariate,
              legend=["1.57nM EGF", "100nM INS", "1.56nM EGF + 100nM INS"],
              saving_path = "",
              cluster_column = cluster_name,
              cluster_name = "Additional information",
              data_type="log2_FC",
              plot_different_data=False,
              saving_info = "",
              save_pdf=False,
              save_png = False,
              plot_close = False,
              y_lims_list=False)

In [ ]:
cluster_number = 7
plot_dataset_phosphosites(df = KShape_all_univariate,
                          cluster_column = cluster_name,
                          cluster_number = cluster_number,
                          data_type = "log2_FC",
                          legend=['1.57 nM EGF', '100 nM Ins', '1.57 nM EGF + 100 nM Ins'],
                          color_palette = ['r', 'b', 'fuchsia'],
                          saving_path="",
                          dataset_name="",
                          saving_info="",
                          plot_individually=False,
                          fit_y_lims=[-1,2],
                          save_pdf=False,
                          save_png=False,
                          plot_close=False
                          )

In [ ]:
KShape_QC = cluster_similarity_per_condition_per_timepoint(df=KShape_all_univariate,
                                                   transpose=True,
                                                   data_type="log2_FC",
                                                   cluster_column_name=cluster_name,
                                                   mean=True,
                                                   median=False,
                                                   verbose = False)
# plot_grouped_bars(KShape_QC)


In [ ]:
plot_cluster_timepoint_dispersion(KShape_QC, ncols=5)


In [ ]:
KShape_QC_over_all = combine_conditions(KShape_QC, how="mean")
plt.bar(x=KShape_QC_over_all.keys(), height=KShape_QC_over_all.values())

## Cluster filter dataframe using HDBSCAN. Exclude "full" time point for clustering



In [ ]:
import hdbscan
from sklearn.preprocessing import StandardScaler

In [ ]:
df_filtered

In [ ]:
column_names = df_filtered.columns.tolist()
condition_for_clustering = ["_EGF_", "_INS_", "EGFnINS"]
data_type = "log2_FC"
exclude_full = False

if len(condition_for_clustering) == 0:
    column_selection = [element for element in column_names if element.startswith(f"{data_type}")] #f"{data_type}" in element]
    if exclude_full == True:
     column_selection = [element for element in column_names if element.startswith(f"{data_type}") and "full" not in element]
else:
    column_selection = [element for element in column_names if element.startswith(f"{data_type}") and any(cond in element for cond in condition_for_clustering)]
    if exclude_full == True:
        column_selection = [element for element in column_names if element.startswith(f"{data_type}") and any(cond in element for cond in condition_for_clustering) and "full" not in element]

print(column_selection)
subset_df = df_filtered[column_selection].copy()
subset_df

# column_names = df_filtered.columns.tolist()
# data_type = "FC_scaled"
#
# if len(condition_for_clustering) == 0:
#     column_selection = [element for element in column_names if element.startswith(f"{data_type}")] #f"{data_type}" in element]
#     if exclude_full == True:
#      column_selection = [element for element in column_names if element.startswith(f"{data_type}") and "full" not in element]
# else:
#     column_selection = [element for element in column_names if element.startswith(f"{data_type}") and any(cond in element for cond in condition_for_clustering)]
#     if exclude_full == True:
#         column_selection = [element for element in column_names if element.startswith(f"{data_type}") and any(cond in element for cond in condition_for_clustering) and "full" not in element]
#
# print(column_selection)
# subset_df_scaled = df_filtered[column_selection].copy()
# subset_df_scaled

In [ ]:
# Standardize across time points
X_scaled = StandardScaler().fit_transform(subset_df)
X_scaled

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=10,
    min_samples=5,
    metric="euclidean"
)

df_filtered_clusters = df_filtered.copy()

labels = clusterer.fit_predict(X_scaled)
len(labels)
df_filtered_clusters["HDBSCAN_cluster_scaled"] = labels #scaled using StandardScaler().

labels_on_log2FC = clusterer.fit_predict(subset_df)
len(labels_on_log2FC)
df_filtered_clusters["HDBSCAN_cluster_log2_FC"] = labels_on_log2FC

labels_on_FC_scaled = clusterer.fit_predict(subset_df)
len(labels_on_FC_scaled)
df_filtered_clusters["HDBSCAN_cluster_FC_scaled"] = labels_on_FC_scaled
df_filtered_clusters

# labels_on_log2FC_EGF = clusterer.fit_predict(subset_df_EGF)
# len(labels_on_log2FC_EGF)
# df_filtered_clusters["HDBSCAN_cluster_log2_FC_EGF"] = labels_on_log2FC_EGF

# subset_df_EGF

In [ ]:
cluster_name = "HDBSCAN_cluster_log2_FC" #"HDBSCAN_cluster_scaled" #"HDBSCAN_cluster_log2_FC"  #"HDBSCAN_cluster_FC_scaled"
clusters_plot(df = df_filtered_clusters,
              legend=["1.57nM EGF", "100nM INS", "1.56nM EGF + 100nM INS"],
              saving_path = "",
              cluster_column = cluster_name,
              cluster_name = "Additional information",
              data_type="log2_FC",
              plot_different_data=True,
              saving_info = "",
              save_pdf=False,
              save_png = False,
              plot_close = False,
              y_lims_list=False)

In [ ]:
cluster_number = 0
plot_dataset_phosphosites(df = df_filtered_clusters,
                          cluster_column = cluster_name,
                          cluster_number = cluster_number,
                          data_type = "log2_FC",
                          legend=['1.57 nM EGF', '100 nM Ins', '1.57 nM EGF + 100 nM Ins'],
                          color_palette = ['r', 'b', 'fuchsia'],
                          saving_path="",
                          dataset_name="",
                          saving_info="",
                          plot_individually=False,
                          fit_y_lims=False,
                          save_pdf=False,
                          save_png=False,
                          plot_close=False
                          )

In [ ]:
HDBSCAN_QC = cluster_similarity_per_condition_per_timepoint(df=df_filtered_clusters,
                                                   transpose=True,
                                                   data_type="FC_scaled",
                                                   cluster_column_name=cluster_name,
                                                   mean=True,
                                                   median=False,
                                                   verbose = False)
plot_cluster_timepoint_dispersion(HDBSCAN_QC, ncols=2)


In [ ]:
HDBSCAN_QC_over_all = combine_conditions(HDBSCAN_QC, how="mean")
plt.bar(x=HDBSCAN_QC_over_all.keys(), height=HDBSCAN_QC_over_all.values())
print(np.mean(list(HDBSCAN_QC_over_all.values())))

# Clustering evaluation. Finding the right amount of clusters
Step 1 — Scan k broadly
Evaluate k in a range (e.g., 5 → 40).
For each k:

- inertia
- silhouette
- stability
- centroid plots

## Inertia (within cluster dispersion)
the sum of distances between each time series and its assigned centroid

Lower inertia = tighter clusters

But inertia always decreases as k increases → use it only to find an elbow.

In [ ]:
df_clustered = df_filtered.copy()
ks = range(2,4)
inertias =[]
for k in ks:
    cluster_name = f"KMeans_{k}_cluster_on_log2FC_noFull_nrep>1_locSite=True_log2FC>0.5"
    print(f"Clustering: {cluster_name}")
    df_clustered, model, multivariate_df = tslearn_clustering_KMeans(df_to_cluster= df_clustered,
                                                 data_type= "log2_FC",
                                                 condition_for_clustering=["_EGF_"], #,"_INS_","_EGFnINS_",],
                                                 exclude_full=True,
                                                 cluster_column_name=cluster_name,
                                                 number_of_clusters=k,
                                                 max_iterations=200,
                                                 n_init = 5,
                                                 metric="euclidean",
                                                 df_dimensions=1,
                                                 time_series_length=6,
                                                 transpose = True,
                                                 verbose=True,
                                                 testing=True)
    inertias.append(model.inertia_)

In [ ]:
df_clustered

In [ ]:
plt.scatter(x=ks, y=inertias)

## Silhouette score (cluster separation)

What silhouette means
For each phosphosite:
How much closer it is to its own cluster than to others
Range:
- (~0.0) → overlapping clusters (common in biology)
- (0.2–0.4) → good for phosphoproteomics
- (>0.5) → rare, usually artificial data


In [ ]:
from tslearn.metrics import cdist_dtw
from sklearn.metrics import silhouette_score

df_clustered = df_filtered.head(1000).copy()
ks = range(2,4)
silhouettes =[]
for k in ks:
    cluster_name = f"KMeans_{k}_cluster_on_log2FC_noFull_nrep>1_locSite=True_log2FC>0.5"
    print(f"Clustering: {cluster_name}")
    df_clustered, model, multivariate = tslearn_clustering_KMeans(df_to_cluster= df_clustered,
                                                 data_type= "log2_FC",
                                                 condition_for_clustering=["_EGF_"], #,"_INS_","_EGFnINS_",],
                                                 exclude_full=True,
                                                 cluster_column_name=cluster_name,
                                                 number_of_clusters=k,
                                                 max_iterations=200,
                                                 n_init = 5,
                                                 metric="euclidean",
                                                 df_dimensions=1,
                                                 time_series_length=6,
                                                 random_state = 0,
                                                 transpose = True,
                                                 verbose=True,
                                                 testing=True)
    # model.labels_
    # print(model.labels_)
    D = cdist_dtw(multivariate)

    sil = silhouette_score(D, model.labels_, metric="precomputed")
    silhouettes.append(sil)




In [ ]:
multivariate

In [ ]:
plt.scatter(x=ks, y=silhouettes)

## Stability

In [ ]:
from sklearn.metrics import adjusted_rand_score

df_clustered = df_filtered.head(1000).copy()
ks = range(2,3)
seeds = range(10)
stabilities =[]

for k in ks:
    labels_list= []
    for seed in seeds:
        cluster_name = f"KMeans_{k}_cluster_seed{seed}_on_log2FC_noFull_nrep>1_locSite=True_log2FC>0.5"
        print(f"Clustering: {cluster_name}")
        df_clustered, model, multivariate = tslearn_clustering_KMeans(df_to_cluster= df_clustered,
                                                 data_type= "log2_FC",
                                                 condition_for_clustering=["_EGF_","_INS_","_EGFnINS_",],
                                                 exclude_full=True,
                                                 cluster_column_name=cluster_name,
                                                 number_of_clusters=k,
                                                 max_iterations=200,
                                                 n_init = 5,
                                                 metric="euclidean",
                                                 df_dimensions=3,
                                                 time_series_length=6,
                                                 random_state = seed,
                                                 transpose = True,
                                                 verbose=True,
                                                 testing=True)

        labels_list.append(model.labels_)

    # pairwise ARI between runs
    ari_scores = []
    for i in range(len(labels_list)):
        for j in range(i+1, len(labels_list)):
            ari_scores.append(
                adjusted_rand_score(labels_list[i], labels_list[j])
            )
    stabilities.append(np.mean(ari_scores))


In [ ]:
# stabilities
plt.scatter(x=ks, y=stabilities)
# plt.figure()
# plt.plot(list(ks), stabilities)
# plt.xlabel("k")
# plt.ylabel("Mean ARI (stability)")
# plt.title("Stability vs k")
# plt.show()

## Centroids

Metrics can mislead.

Centroid plots answer the real question:

Are these genuinely different kinetic shapes?

This is biologically decisive.

**Check for cluster numbaers candidates**

In [ ]:
k = 2
cluster_name = f"KMeans_{k}_cluster_seed0_on_log2FC_noFull_nrep>1_locSite=True_log2FC>0.5" #carefull with the "seed0"

clusters_plot_linear(
    df = df_clustered,
    legend=["1.57nM EGF", "100nM INS", "1.56nM EGF + 100nM INS"],
    saving_path="",
    cluster_column=cluster_name,
    cluster_name="chaecking centroids",
    data_type="log2_FC",
    plot_different_data=True,
    saving_info="dd",
    save_pdf=False,
    save_png=False,
    plot_close=False,
    y_lims_list=False,
    grey_alpha=0.2,
    grey_lw=0.8,
    mean_lw=2.6
)

In [ ]:
k = 2
cluster_name = f"KMeans_{k}_cluster_seed0_on_log2FC_noFull_nrep>1_locSite=True_log2FC>0.5" #carefull with the "seed0"

clusters_plot_overlay_conditions(
    df = df_clustered,
    legend=["1.57nM EGF", "100nM INS", "1.56nM EGF + 100nM INS"],
    saving_path="",
    cluster_column=cluster_name,
    cluster_name="testing",
    data_type="log2_FC",
    plot_different_data=True,
    saving_info="",
    save_pdf=False,
    save_png=False,
    plot_close=False,
    y_lims_list=False,
    individual_alpha=0.08,
    individual_lw=0.8,
    mean_lw=2.8
)


## Inertia and silhouette score together

In [ ]:
from tslearn.metrics import cdist_dtw
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score


df_clustered = df_filtered.head(500).copy()

ks = range(2,3)
seeds = range(2)

inertias =[]
silhouettes =[]
stabilities =[]

for k in ks:

    labels_list= []
    for seed in seeds:
        cluster_name = f"KMeans_{k}_cluster_seed{seed}_on_log2FC_noFull_nrep>1_locSite=True_log2FC>0.5"
        print(f"Clustering: {cluster_name}")

        df_clustered, model, multivariate = tslearn_clustering_KMeans(df_to_cluster= df_clustered,
                                                     data_type= "log2_FC",
                                                     condition_for_clustering=["_EGF_", "_INS_","_EGFnINS_",],
                                                     exclude_full=True,
                                                     cluster_column_name=cluster_name,
                                                     number_of_clusters=k,
                                                     max_iterations=200,
                                                     n_init = 5,
                                                     metric="euclidean",
                                                     df_dimensions=3,
                                                     time_series_length=6,
                                                     random_state = seed,
                                                     transpose = True,
                                                     verbose=True,
                                                     testing=True)

        if seed == 0:
            inertias.append(model.inertia_)

            D = cdist_dtw(multivariate)
            sil = silhouette_score(D, model.labels_, metric="euclidean")
            silhouettes.append(sil)

        labels_list.append(model.labels_)

    # pairwise ARI between runs
    ari_scores = []
    for i in range(len(labels_list)):
        for j in range(i+1, len(labels_list)):
            ari_scores.append(
                adjusted_rand_score(labels_list[i], labels_list[j])
            )
    stabilities.append(np.mean(ari_scores))


plt.scatter(x=ks, y=inertias)
plt.title("Inertia vs ks")
plt.savefig(f"inertias.png")
plt.close()

#silhouetes
plt.scatter(x=ks, y=silhouettes)
plt.title("Silhouettes vs ks")
plt.savefig(f"silhouettes.png")
plt.close()

#stabilities
plt.scatter(x=ks, y=stabilities)
plt.title("Stabilities vs ks")
plt.savefig(f"silhouettes.png")
plt.close()

In [ ]:
plt.scatter(x=ks, y=inertias)
plt.title("Inertia vs clusters")
plt.xlabel("Inertia")
plt.ylabel("Clusters")

In [ ]:
plt.scatter(x=ks, y=inertias)

In [ ]:
plt.scatter(x=ks, y=silhouettes)

In [ ]:
plt.scatter(x=ks, y=stabilities)

In [ ]:
k = 40
seed = 0
cluster_name = f"KMeans_{k}_cluster_seed{seed}_on_log2FC_noFull_nrep>1_locSite=True_log2FC>0.5" #carefull with the "seed0"

clusters_plot_linear(
    df = df_clustered,
    legend=["1.57nM EGF", "100nM INS", "1.56nM EGF + 100nM INS"],
    saving_path="",
    cluster_column=cluster_name,
    cluster_name="chaecking centroids",
    data_type="log2_FC",
    plot_different_data=True,
    saving_info="dd",
    save_pdf=False,
    save_png=False,
    plot_close=False,
    y_lims_list=False,
    grey_alpha=0.2,
    grey_lw=0.8,
    mean_lw=2.6
)

# Analyzing output.log file server

In [ ]:
import re
import matplotlib.pyplot as plt

# Storage lists
k_values = []
inertia_values = []
silhouette_values = []
stability_values =  []

pattern = re.compile(
    r'k\s*=\s*([-+]?\d*\.?\d+).*?inertia?\s*=\s*([-+]?\d*\.?\d+).*?silhouette?\s*=\s*([-+]?\d*\.?\d+)..*?stability?\s*=\s*([-+]?\d*\.?\d+)'
)

with open("Server_results/output_EGF.log", "r") as f:
    for line in f:
        match = pattern.search(line)
        if match:
            k_values.append(float(match.group(1)))
            inertia_values.append(float(match.group(2)))
            silhouette_values.append(float(match.group(3)))
            stability_values.append(float(match.group(4)))
# Sanity check
# print("K values:", k_values)
# print("Inertias:", inertia_values)

assert len(k_values) == len(inertia_values) == len(silhouette_values) == len(stability_values), "Mismatch in extracted data"

# Take only a subset for plotting
subset = 20
k_values = k_values[:subset]
inertia_values = inertia_values[:subset]
silhouette_values = silhouette_values[:subset]
stability_values = stability_values[:subset]

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].plot(k_values, inertia_values, marker='o')
ax[0].set_title("Inertia")
ax[0].set_xlabel("Clusters")
ax[0].set_ylabel("Inertia")

ax[1].plot(k_values, silhouette_values, marker='o')
ax[1].set_title("Silhouette")
ax[1].set_xlabel("Clusters")
ax[1].set_ylabel("Silhouette Score")

ax[2].plot(k_values, stability_values, marker='o')
ax[2].set_title("Stability")
ax[2].set_xlabel("Clusters")
ax[2].set_ylabel("Stability")

fig.suptitle("Clustering evaluation for multivariate time series (excludding 'full')", fontsize=16)
fig.tight_layout()
# fig.savefig(f"Experiment/2_hTERT_HME1/Results/Clusters_QC/KMeans/EGF_20_clustering_evaluation.png")
fig.show()


# Manually merging clusters

In [ ]:
nk = 25
cluster_name = f"KMeans_{nk}_cluster_on_log2FC_noFull_nrep>1_locSite=True_log2FC>0.5_and_manually_merged"
kmeans_clustered= tslearn_clustering_KMeans(df_to_cluster= df_filtered,
                                             data_type= "log2_FC",
                                             condition_for_clustering=["_EGF_", "_INS_", "_EGFnINS"],
                                             exclude_full=True,
                                             cluster_column_name=cluster_name,
                                             number_of_clusters=nk,
                                             max_iterations=100,
                                             metric="euclidean",
                                             df_dimensions=3,
                                             time_series_length=6,
                                             transpose = True, #data size is (ns, sz, d)
                                             verbose=True,
                                             testing = False,
                                             barycenter_calculations= False)

In [ ]:
clusters_plot_linear(
    df = kmeans_clustered,
    legend=["1.57nM EGF", "100nM INS", "1.56nM EGF + 100nM INS"],
    saving_path="",
    cluster_column=cluster_name,
    cluster_name="Checking centroids",
    data_type="log2_FC",
    plot_different_data=True,
    saving_info="",
    save_pdf=False,
    save_png=False,
    plot_close=False,
    y_lims_list=False,
    grey_alpha=0.2,
    grey_lw=0.8,
    mean_lw=2.6
)

In [ ]:
kmeans_clustered[cluster_name] = kmeans_clustered[cluster_name].replace({6:2, 11:2,
                                                                         12:9, 17:9,
                                                                         24:22,
                                                                         19:0,
                                                                         16:1})



In [ ]:
clusters_plot_linear(
    df = kmeans_clustered,
    legend=["1.57nM EGF", "100nM INS", "1.56nM EGF + 100nM INS"],
    saving_path="Experiment/2_hTERT_HME1/Results/Clusters_QC/KMeans",
    cluster_column=cluster_name,
    cluster_name="Manually mergin clusters",
    data_type="log2_FC",
    plot_different_data=True,
    saving_info="lo2FC_manually_merged",
    save_pdf=False,
    save_png=True,
    plot_close=False,
    y_lims_list=False,
    grey_alpha=0.2,
    grey_lw=0.8,
    mean_lw=2.6
)

In [ ]:
# kmeans_clustered[cluster_name] = pd.factorize(kmeans_clustered[cluster_name])[0]

clusters_plot_linear(
    df = kmeans_clustered,
    legend=["1.57nM EGF", "100nM INS", "1.56nM EGF + 100nM INS"],
    saving_path="",
    cluster_column=cluster_name,
    cluster_name="Checking centroids",
    data_type="log2_FC",
    plot_different_data=True,
    saving_info="",
    save_pdf=False,
    save_png=False,
    plot_close=False,
    y_lims_list=False,
    grey_alpha=0.2,
    grey_lw=0.8,
    mean_lw=2.6
)

In [ ]:
manually_merged_df = kmeans_clustered.copy()

In [ ]:
manually_merged_df

In [ ]:

column_names = manually_merged_df.columns.tolist()
column_selection = [element for element in column_names if element.startswith(f"log2_FC") and any(cond in element for cond in ["_EGF_", "_INS_", "_EGFnINS_"]) and "full" not in element]
column_selection

In [ ]:
manually_merged_np, labels = reshape_df(manually_merged_df, time_series = column_selection, dimensions = 3, len_time_serie = 6, transpose=True, labels = "site", verbose=True)

In [ ]:
from tslearn.barycenters import dtw_barycenter_averaging

nk = 25
cluster_name = f"KMeans_{nk}_cluster_on_log2FC_noFull_nrep>1_locSite=True_log2FC>0.5_and_manually_merged"

clusters = manually_merged_df[cluster_name].unique()
new_centroids = []

for k in clusters:
    X_k = manually_merged_np[manually_merged_df[cluster_name].values == k]
    centroid_k = dtw_barycenter_averaging(X_k)
    new_centroids.append(centroid_k)

# new_centroids = np.array(new_centroids)
#
new_centroids = np.array([
    manually_merged_np[manually_merged_df[cluster_name].values == k].mean(axis=0)
    for k in clusters
])

In [ ]:
from tslearn.metrics import cdist_dtw

D_new = cdist_dtw(manually_merged_np, new_centroids)

In [ ]:
sorted_D = np.sort(D_new, axis=1)
margin = sorted_D[:, 1] - sorted_D[:, 0]

In [ ]:
plt.figure()
plt.hist(margin, bins=30)
plt.xlabel("Margin = 2nd_best - best")
plt.ylabel("Count")
plt.title("Cluster assignment margin distribution (merged)")
plt.show()

In [ ]:
labels_new = D_new.argmin(axis=1)

plt.figure()
plt.boxplot(
    [margin[labels_new == k] for k in range(len(clusters))],
    #tick_labels=[str(k) for k in range(len(clusters))]
)
plt.xlabel("Cluster")
plt.ylabel("Margin")
plt.title("Assignment margin per cluster (merged)")
plt.show()

In [ ]:
Z = PCA(n_components=2).fit_transform(D_new)

plt.figure()
plt.scatter(Z[:, 0], Z[:, 1], c=labels_new)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of distance-to-centroid representation (merged)")
plt.show()

In [ ]:
# Sanity chcek
np.all(labels_new == manually_merged_df[cluster_name].values)

# Clustering sub_Clusters

In [ ]:
#from 45 clusters by FCscaled, subcluster cluster 2
cluster_name
subset_cluster2 = kmeans_clustered.loc[kmeans_clustered[cluster_name] == 2]

In [ ]:
subcluster_name = f"KMeans_25_subcluster2_cluster_on_FCscaled_noFull_nrep>1_locSite=True_log2FC>0.5_transform"
subclustering = tslearn_clustering_KMeans(df_to_cluster= subset_cluster2,
                                             data_type= "FC_scaled",
                                             condition_for_clustering=["_EGF_", "_INS_", "_EGFnINS"],
                                             exclude_full=True,
                                             cluster_column_name=subcluster_name,
                                             number_of_clusters=3,
                                             max_iterations=100,
                                             metric="euclidean",
                                             df_dimensions=3,
                                             time_series_length=6,
                                             transpose = True, #data size is (ns, sz, d)
                                             verbose=True,
                                             testing = False,
                                             barycenter_calculations= False)
clusters_plot_linear(
        df = subclustering,
        legend=["1.57nM EGF", "100nM INS", "1.56nM EGF + 100nM INS"],
        saving_path="Experiment/2_hTERT_HME1/Results/Clusters_QC/KMeans",
        cluster_column=subcluster_name,
        cluster_name=f"{nk}_clusters",
        data_type="FC_scaled",
        plot_different_data=True,
        saving_info="_FCscaled_Shapes",
        save_pdf=False,
        save_png=False,
        plot_close=False,
        y_lims_list=False,
        grey_alpha=0.2,
        grey_lw=0.8,
        mean_lw=2.6
    )